# Tech Challenge Fase 01 — Etapa 2
## Modelagem com Rede Neural (MLP em PyTorch)

O objetivo dessa etapa é construir, treinar e avaliar uma rede neural do tipo MLP para o problema de previsão de churn, comparando seu desempenho com os baselines desenvolvidos na Etapa 1.

Serão utilizados:
- PyTorch para construção da rede neural;
- métricas adequadas ao problema de classificação desbalanceada;
- early stopping e batching no treinamento;
- MLflow para rastreamento dos experimentos.

## Preparação dos dados para PyTorch

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# seed para reprodutibilidade
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
import torch

print(f"Versão do PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")

In [ ]:
# carregar a base
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

FILE_NAME = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
file_path = RAW_DIR / FILE_NAME

df = pd.read_csv(file_path)
df.head()

In [ ]:
# tratamento
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

df = df.drop(columns=["customerID"])

In [ ]:
# separar x e y
X = df.drop(columns=["Churn"])
y = (df["Churn"] == "Yes").astype(int)

print(X.shape, y.shape)
print(y.value_counts(normalize=True).round(4))

In [ ]:
# treino, validação e teste
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=SEED, stratify=y_train_full
)

print("Treino:", X_train.shape, y_train.shape)
print("Validação:", X_val.shape, y_val.shape)
print("Teste:", X_test.shape, y_test.shape)

In [ ]:
# pré-processamento
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

In [ ]:
# transformar os dados

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_val_processed.shape)
print(X_test_processed.shape)

In [ ]:
# converter para tensores
X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_processed, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [ ]:
# criar DataLoaders
BATCH_SIZE = 64

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Definição da arquitetura da MLP

In [ ]:
# input dimension
input_dim = X_train_processed.shape[1]
input_dim

In [ ]:
# defininfo MLP
class MLPChurn(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# criar instância
model = MLPChurn(input_dim=input_dim).to(device)
model

In [ ]:
# definindo loss e otimizador
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# teste rápido
sample_batch = next(iter(train_loader))
X_sample, y_sample = sample_batch

X_sample = X_sample.to(device)
y_sample = y_sample.to(device)

outputs = model(X_sample)
print("Shape da saída:", outputs.shape)

## Treinamento com early stopping

In [ ]:
# hiperparâmetros
EPOCHS = 100
PATIENCE = 10

# listas para histórico
train_losses = []
val_losses = []

best_val_loss = float("inf")
epochs_without_improvement = 0
best_model_state = None

In [ ]:
# loop de treinamento
for epoch in range(EPOCHS):
    # =========================
    # TREINAMENTO
    # =========================
    model.train()
    running_train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * X_batch.size(0)

    epoch_train_loss = running_train_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    # =========================
    # VALIDAÇÃO
    # =========================
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            running_val_loss += loss.item() * X_batch.size(0)

    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f}"
    )

    # =========================
    # EARLY STOPPING
    # =========================
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        epochs_without_improvement = 0
        best_model_state = model.state_dict()
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping ativado na época {epoch + 1}.")
        break

In [ ]:
# restaurar melhor modelo
if best_model_state is not None:
    model.load_state_dict(best_model_state)

In [ ]:
# gráfico das losses
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Épocas")
plt.ylabel("Loss")
plt.title("Curvas de Treinamento e Validação")
plt.legend()
plt.show()

In [ ]:
# gerar probabilidades no conjunto de teste
model.eval()

all_probs = []
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        probs = torch.sigmoid(logits)

        preds = (probs >= 0.5).float()

        all_probs.extend(probs.cpu().numpy().flatten())
        all_preds.extend(preds.cpu().numpy().flatten())
        all_targets.extend(y_batch.cpu().numpy().flatten())

In [ ]:
# converter para arrays numpy
all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

print(all_probs[:5])
print(all_preds[:5])
print(all_targets[:5])

In [ ]:
# calcular métricas da MLP
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

mlp_accuracy = accuracy_score(all_targets, all_preds)
mlp_precision = precision_score(all_targets, all_preds, zero_division=0)
mlp_recall = recall_score(all_targets, all_preds, zero_division=0)
mlp_f1 = f1_score(all_targets, all_preds, zero_division=0)
mlp_roc_auc = roc_auc_score(all_targets, all_probs)
mlp_pr_auc = average_precision_score(all_targets, all_probs)

print("Accuracy:", round(mlp_accuracy, 4))
print("Precision:", round(mlp_precision, 4))
print("Recall:", round(mlp_recall, 4))
print("F1-score:", round(mlp_f1, 4))
print("ROC-AUC:", round(mlp_roc_auc, 4))
print("PR-AUC:", round(mlp_pr_auc, 4))

In [ ]:
# matriz de confusão
cm = confusion_matrix(all_targets, all_preds)
print(cm)

## Avaliação da MLP

A MLP apresentou um desempenho competitivo na tarefa de previsão de churn e conseguiu identificar uma parcela relevante dos clientes que realmente cancelaram.

Em comparação com a Regressão Logística, a rede neural teve recall um pouco maior, o que significa que ela conseguiu capturar mais clientes com risco de evasão. Em contrapartida, a precision foi menor, indicando um aumento na quantidade de falsos positivos.

As métricas de F1-score, ROC-AUC e PR-AUC ficaram muito próximas das obtidas pela Regressão Logística. Isso sugere que, nesta configuração inicial, a MLP não foi claramente superior ao baseline logístico, mas trouxe um comportamento diferente na forma de equilibrar os erros do modelo.

## Comparação com modelos baseline

In [ ]:
# imports dos baselines
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline

In [ ]:
# preparar preprocessor para os baselines
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

baseline_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

In [ ]:
# DummyClassifier
dummy_model = Pipeline(
    steps=[
        ("preprocessor", baseline_preprocessor),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]
)

dummy_model.fit(X_train_full, y_train_full)

y_pred_dummy = dummy_model.predict(X_test)

# Logistic Regression
log_reg_model = Pipeline(
    steps=[
        ("preprocessor", baseline_preprocessor),
        ("classifier", LogisticRegression(random_state=SEED, max_iter=1000)),
    ]
)

log_reg_model.fit(X_train_full, y_train_full)

y_pred_log = log_reg_model.predict(X_test)
y_proba_log = log_reg_model.predict_proba(X_test)[:, 1]

In [ ]:
results_all_models = pd.DataFrame(
    [
        {
            "Model": "DummyClassifier",
            "Accuracy": accuracy_score(y_test, y_pred_dummy),
            "Precision": precision_score(y_test, y_pred_dummy, zero_division=0),
            "Recall": recall_score(y_test, y_pred_dummy, zero_division=0),
            "F1-score": f1_score(y_test, y_pred_dummy, zero_division=0),
            "ROC-AUC": np.nan,
            "PR-AUC": np.nan,
        },
        {
            "Model": "LogisticRegression",
            "Accuracy": accuracy_score(y_test, y_pred_log),
            "Precision": precision_score(y_test, y_pred_log, zero_division=0),
            "Recall": recall_score(y_test, y_pred_log, zero_division=0),
            "F1-score": f1_score(y_test, y_pred_log, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_proba_log),
            "PR-AUC": average_precision_score(y_test, y_proba_log),
        },
        {
            "Model": "MLP (PyTorch)",
            "Accuracy": mlp_accuracy,
            "Precision": mlp_precision,
            "Recall": mlp_recall,
            "F1-score": mlp_f1,
            "ROC-AUC": mlp_roc_auc,
            "PR-AUC": mlp_pr_auc,
        },
    ]
)

results_all_models.round(4)

## Interpretação da comparação

Comparando os modelos, ficou claro que tanto a Regressão Logística quanto a MLP foram muito superiores ao baseline ingênuo.

A Regressão Logística teve melhor accuracy e precision, o que indica um comportamento mais conservador e maior proporção de acertos entre os clientes previstos como churn.

Já a MLP apresentou recall mais alto, o que mostra maior capacidade de identificar clientes que realmente cancelam. Em compensação, isso veio junto com uma queda em precision, ou seja, mais clientes foram marcados como churn sem realmente cancelar.

Como as métricas de F1-score, ROC-AUC e PR-AUC ficaram bem próximas, a principal diferença entre os dois modelos não está em uma superioridade clara de desempenho, mas sim no tipo de trade-off que cada um oferece.

## Registro no MLflow

In [ ]:
from pathlib import Path

import mlflow

# Raiz do projeto: notebooks/../ = project root (CWD do Jupyter = dir do notebook)
_project_root = Path.cwd().parent
mlflow.set_tracking_uri(f"sqlite:///{_project_root / 'mlflow.db'}")

# experimento da Etapa 2
mlflow.set_experiment("tech_challenge_fiap_fase1_etapa2_mlp")

# salvar o gráfico de loss
loss_plot_path = "mlp_training_curve.png"

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Épocas")
plt.ylabel("Loss")
plt.title("Curvas de Treinamento e Validação - MLP")
plt.legend()
plt.savefig(loss_plot_path, bbox_inches="tight")
plt.show()

In [ ]:
# registrar MLP MLflow
with mlflow.start_run(run_name="mlp_pytorch_baseline"):
    # Parâmetros do modelo
    mlflow.log_param("model", "MLP_PyTorch")
    mlflow.log_param("input_dim", input_dim)
    mlflow.log_param("hidden_layer_1", 64)
    mlflow.log_param("hidden_layer_2", 32)
    mlflow.log_param("activation", "ReLU")
    mlflow.log_param("loss_function", "BCEWithLogitsLoss")
    mlflow.log_param("optimizer", "Adam")
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("epochs_max", EPOCHS)
    mlflow.log_param("patience", PATIENCE)
    mlflow.log_param("seed", SEED)

    # Dataset / contexto
    mlflow.log_param("dataset_name", "WA_Fn-UseC_-Telco-Customer-Churn.csv")
    mlflow.log_param("task", "binary_classification_churn")

    # Métricas finais
    mlflow.log_metric("accuracy", mlp_accuracy)
    mlflow.log_metric("precision", mlp_precision)
    mlflow.log_metric("recall", mlp_recall)
    mlflow.log_metric("f1_score", mlp_f1)
    mlflow.log_metric("roc_auc", mlp_roc_auc)
    mlflow.log_metric("pr_auc", mlp_pr_auc)
    mlflow.log_metric("best_val_loss", best_val_loss)
    mlflow.log_metric("epochs_ran", len(train_losses))

    # Artefato: gráfico de loss
    mlflow.log_artifact(loss_plot_path)

    # Artefato: state_dict do modelo
    model_path = "mlp_model_state_dict.pth"
    torch.save(model.state_dict(), model_path)
    mlflow.log_artifact(model_path)

## Conclusão da Etapa 2

Nesta etapa, foi construída uma rede neural do tipo MLP em PyTorch para o problema de previsão de churn, utilizando batching, função de perda adequada para classificação binária, otimizador Adam e early stopping com base na perda de validação.

Os resultados mostraram que a MLP teve desempenho competitivo em relação à Regressão Logística. O principal destaque foi o recall mais alto, indicando maior capacidade de identificar clientes com risco de evasão. Por outro lado, a precision foi menor, o que mostra que esse ganho veio acompanhado de mais falsos positivos.

De forma geral, as métricas de F1-score, ROC-AUC e PR-AUC ficaram muito próximas entre os dois modelos. Isso indica que, nesta configuração inicial, a MLP não superou claramente o baseline logístico, mas apresentou um trade-off diferente, que pode ser interessante dependendo da estratégia da empresa.

Por fim, os resultados e artefatos da MLP foram registrados no MLflow, o que ajuda a manter a rastreabilidade dos experimentos e facilita a continuidade do projeto nas próximas etapas.